In [ ]:
# Cell 1: Install project dependencies
%pip -q install -U crewai gradio

In [ ]:
# Cell 2: Imports and application configuration

import os
import re
import json
from datetime import datetime

import gradio as gr
from google.colab import userdata
from crewai import Agent, Task, Crew, Process, LLM

APP_NAME = "Leo — Multi-Agent AI Tutor"

# Use the exact stable Gemini model ID.
MODEL_NAME = "gemini/gemini-3.1-flash-lite"

TEMPERATURE = 0.2
MAX_RETEACH_ATTEMPTS = 2

print("Project configuration loaded.")
print("Model:", MODEL_NAME)

Project configuration loaded.
Model: gemini/gemini-3.1-flash-lite


In [ ]:
# Cell 3: Securely load the Gemini API key

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY is missing. "
        "Add it in Google Colab Secrets and enable notebook access."
    )

# CrewAI's Gemini provider reads the key from the environment.
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# Initialize the Gemini model through CrewAI.
gemini_llm = LLM(
    model=MODEL_NAME,
    api_key=GEMINI_API_KEY,
    temperature=TEMPERATURE,
)

print("API key loaded securely; its value is not displayed.")

API key loaded securely; its value is not displayed.


In [ ]:
import asyncio
# Cell 4: Verify Gemini model access before building the tutor

test_agent = Agent(
    role="Connection Tester",
    goal="Test whether the configured Gemini model can generate a response.",
    backstory="You are a concise AI system tester.",
    llm=gemini_llm,
    verbose=False,
    allow_delegation=False,
    max_iter=2,
)

test_task = Task(
    description="Explain a Python variable in exactly two simple sentences.",
    expected_output="Two clear sentences.",
    agent=test_agent,
)

async def run_test_crew():
    try:
        test_crew = Crew(
            agents=[test_agent],
            tasks=[test_task],
            process=Process.sequential,
            verbose=False,
        )
        test_result = await test_crew.kickoff_async()
        print("Gemini connection successful.")
        print(test_result)

    except Exception as error:
        print("Model connection failed.")
        print(type(error).__name__, str(error)[:800])
        raise

await run_test_crew()

Gemini connection successful.
A Python variable acts as a named container used to store data values in a program. You can assign information to these labels and reference them later by their specific names.


In [ ]:
# Cell 5: Reusable agent utility and session memory

def make_agent(role, goal, backstory):
    """Create a specialized CrewAI agent using the configured Gemini model."""
    return Agent(
        role=role,
        goal=goal,
        backstory=backstory,
        llm=gemini_llm,
        verbose=False,
        allow_delegation=False,
        max_iter=3,
    )


def run_single_agent(agent, description, expected_output):
    """Run one specialized agent task and return its output as text."""
    task = Task(
        description=description,
        expected_output=expected_output,
        agent=agent,
    )
    crew = Crew(
        agents=[agent],
        tasks=[task],
        process=Process.sequential,
        verbose=False,
    )
    return str(crew.kickoff())


def new_memory():
    """Create isolated session memory for one student."""
    return {
        "student_name": "",
        "topic": "",
        "difficulty": "Beginner",
        "learning_plan": "",
        "lesson": "",
        "quiz": [],
        "answers": [],
        "score": None,
        "feedback": "",
        "weak_areas": [],
        "reteaching": "",
        "reteach_attempts": 0,
        "history": [],
    }


print("Agent utility and student memory are ready.")

Agent utility and student memory are ready.


In [ ]:
# Cell 6: Coordinator Agent

coordinator = make_agent(
    role="Learning Coordinator",
    goal="Analyze the student's request and create a structured learning plan.",
    backstory=(
        "You are an educational coordinator who identifies learning "
        "objectives, prerequisites, ordered subtopics and measurable outcomes."
    ),
)


def coordinator_plan(name, topic, difficulty):
    prompt = f"""
    Create a personalized learning plan.

    Student: {name}
    Topic: {topic}
    Level: {difficulty}

    Include:
    1. Learning objective.
    2. Three to five subtopics in logical order.
    3. Prerequisites if needed.
    4. Practical learning outcome.
    5. How understanding will be assessed.

    Keep the plan clear and suitable for the student's level.
    Do not generate quiz questions yet.
    """
    return run_single_agent(
        coordinator,
        prompt,
        "A structured learning plan with objectives and ordered subtopics.",
    )


print("Coordinator Agent ready.")

Coordinator Agent ready.


In [ ]:
# Cell 7: Explainer Agent

explainer = make_agent(
    role="Concept Explainer",
    goal="Teach the requested topic with accurate explanations and examples.",
    backstory=(
        "You are a patient computer science tutor. You break complex "
        "concepts into understandable steps and use practical examples."
    ),
)


def explain_topic(name, topic, difficulty, plan):
    prompt = f"""
    Teach the student using the Coordinator's learning plan.

    Student: {name}
    Topic: {topic}
    Level: {difficulty}

    LEARNING PLAN:
    {plan}

    Structure the lesson with:
    1. Introduction.
    2. Step-by-step explanations.
    3. At least two practical examples where appropriate.
    4. Common mistakes.
    5. Short recap.

    Use clear Markdown headings and lists.
    Do not reveal answers to the upcoming quiz.
    """
    return run_single_agent(
        explainer,
        prompt,
        "A complete lesson with explanations, examples and a recap.",
    )


print("Explainer Agent ready.")

Explainer Agent ready.


In [ ]:
# Cell 8: Quiz Master Agent and JSON validation

quiz_master = make_agent(
    role="Quiz Master",
    goal="Create exactly five clear MCQs based on the lesson.",
    backstory=(
        "You are an assessment designer. Every question has four options, "
        "one correct answer, a topic label and an explanation."
    ),
)


def extract_json(text):
    """Parse a JSON object from the model output."""
    cleaned = str(text).strip()
    cleaned = re.sub(
        r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE
    )
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        data = json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
        if not match:
            raise ValueError("No valid JSON object found in quiz output.")
        data = json.loads(match.group(0))

    if not isinstance(data, dict):
        raise ValueError("Quiz output must be a JSON object.")

    return data


def validate_quiz(data):
    """Check the exact quiz structure before showing it to the student."""
    questions = data.get("questions")

    if not isinstance(questions, list) or len(questions) != 5:
        raise ValueError("Quiz Master must return exactly five questions.")

    for index, question in enumerate(questions, start=1):
        if not isinstance(question, dict):
            raise ValueError(f"Question {index} is not a JSON object.")

        question["question"] = str(question.get("question", "")).strip()
        question["topic"] = str(
            question.get("topic", "General")
        ).strip()
        question["explanation"] = str(
            question.get("explanation", "")
        ).strip()

        options = question.get("options", {})

        if isinstance(options, list) and len(options) == 4:
            options = dict(zip(["A", "B", "C", "D"], options))

        if not isinstance(options, dict):
            raise ValueError(f"Question {index} has invalid options.")

        if not all(letter in options for letter in ["A", "B", "C", "D"]):
            raise ValueError(f"Question {index} needs options A, B, C and D.")

        question["options"] = {
            letter: str(options[letter]).strip()
            for letter in ["A", "B", "C", "D"]
        }

        question["answer"] = str(
            question.get("answer", "")
        ).strip().upper()

        if question["answer"] not in {"A", "B", "C", "D"}:
            raise ValueError(f"Question {index} has an invalid answer key.")

        if not question["question"]:
            raise ValueError(f"Question {index} has empty question text.")

    return questions


def generate_quiz(name, topic, difficulty, lesson):
    prompt = f"""
    Create exactly FIVE MCQs based on the lesson below.

    Student: {name}
    Topic: {topic}
    Difficulty: {difficulty}

    LESSON:
    {lesson}

    Requirements:
    - Exactly five questions, progressing from easy to challenging.
    - Four options per question: A, B, C and D.
    - Exactly one correct option per question.
    - Each question must include its topic and a short explanation.
    - Do not include the correct answer in the question text.
    - Do not add text outside the JSON object.

    Return this JSON structure:
    {{
      "questions": [
        {{
          "question": "Question text",
          "options": {{
            "A": "Option A",
            "B": "Option B",
            "C": "Option C",
            "D": "Option D"
          }},
          "answer": "A",
          "topic": "Concept tested",
          "explanation": "Why the answer is correct"
        }}
      ]
    }}
    """
    raw = run_single_agent(
        quiz_master,
        prompt,
        "A valid JSON object containing exactly five MCQs.",
    )
    return validate_quiz(extract_json(raw))


print("Quiz Master ready. Output will be validated before display.")

Quiz Master ready. Output will be validated before display.


In [ ]:
# Cell 9: Evaluator Agent and deterministic scoring

evaluator = make_agent(
    role="Student Evaluator",
    goal="Give constructive feedback based on verified quiz results.",
    backstory=(
        "You are a supportive academic evaluator. You identify strengths, "
        "misconceptions and weak topics using the submitted answers."
    ),
)


def score_quiz(quiz, answers):
    """Calculate the exact score and list weak topics."""
    if len(quiz) != 5 or len(answers) != 5:
        raise ValueError("Exactly five questions and answers are required.")

    score = 0
    question_results = []
    weak_areas = []

    for number, (question, answer) in enumerate(zip(quiz, answers), start=1):
        submitted = str(answer or "").strip().upper()
        correct = question["answer"]
        is_correct = submitted == correct

        if is_correct:
            score += 1
        else:
            weak_areas.append(question["topic"])

        question_results.append({
            "number": number,
            "topic": question["topic"],
            "submitted": submitted or "No answer",
            "correct_answer": correct,
            "is_correct": is_correct,
            "explanation": question["explanation"],
        })

    return {
        "score": score,
        "total": 5,
        "percentage": score * 20,
        "weak_areas": list(dict.fromkeys(weak_areas)),
        "question_results": question_results,
    }


def evaluate_answers(name, topic, lesson, results):
    """Generate feedback without allowing the model to alter the score."""
    prompt = f"""
    Evaluate this student's quiz performance.

    Student: {name}
    Topic: {topic}
    Verified score: {results['score']}/5
    Percentage: {results['percentage']}%
    Weak areas: {results['weak_areas']}

    Verified question results:
    {json.dumps(results['question_results'], indent=2)}

    Lesson context:
    {lesson}

    Provide:
    1. A performance summary using the verified score.
    2. Feedback on incorrect answers.
    3. Concepts understood correctly.
    4. Specific topics that need review.
    5. Encouraging next steps.

    Never change the verified score or invent answer results.
    """
    return run_single_agent(
        evaluator,
        prompt,
        "Constructive feedback based on the verified quiz results.",
    )


print("Evaluator ready.")

Evaluator ready.


In [ ]:
# Cell 10: Targeted re-teaching for weak areas

reteaching_agent = make_agent(
    role="Targeted Re-teaching Specialist",
    goal="Teach weak concepts again using alternative explanations and examples.",
    backstory=(
        "You are a patient remedial tutor. You focus on actual mistakes "
        "and teach difficult concepts in smaller, easier steps."
    ),
)


def reteach_weak_areas(name, topic, weak_areas, question_results):
    """Create a targeted mini-lesson from the evaluation."""
    if not weak_areas:
        return (
            "Excellent! All five answers were correct. "
            "No re-teaching is needed; try a harder topic or quiz."
        )

    incorrect = [
        item for item in question_results
        if not item["is_correct"]
    ]

    prompt = f"""
    Create a targeted re-teaching lesson.

    Student: {name}
    Main topic: {topic}
    Weak areas: {weak_areas}

    Incorrect answer details:
    {json.dumps(incorrect, indent=2)}

    Include:
    1. Simple explanations of the weak concepts.
    2. A different example or analogy for each concept.
    3. Why the incorrect answer may be confusing.
    4. One short practice activity per weak concept.
    5. A concise recap.

    Focus only on the identified weak areas.
    Do not reveal unrelated quiz answers.
    """
    return run_single_agent(
        reteaching_agent,
        prompt,
        "A targeted mini-lesson with examples, practice and recap.",
    )


print("Re-teaching Agent ready.")

Re-teaching Agent ready.


In [ ]:
# Cell 11: Main workflow — Coordinator -> Explainer -> Quiz Master

def start_session(student_name, topic, difficulty):
    """Run the three initial agents and store their results in memory."""
    name = str(student_name or "").strip()
    topic = str(topic or "").strip()

    if not name:
        raise ValueError("Enter your name first.")
    if not topic:
        raise ValueError("Enter a topic to learn.")

    memory = new_memory()
    memory.update({
        "student_name": name,
        "topic": topic,
        "difficulty": difficulty,
    })

    activity = []

    activity.append("1/3 Coordinator: preparing the learning plan.")
    memory["learning_plan"] = coordinator_plan(name, topic, difficulty)

    activity.append("2/3 Explainer: teaching the topic.")
    memory["lesson"] = explain_topic(
        name, topic, difficulty, memory["learning_plan"]
    )

    activity.append("3/3 Quiz Master: generating five MCQs.")
    memory["quiz"] = generate_quiz(
        name, topic, difficulty, memory["lesson"]
    )

    memory["history"].append({
        "event": "session_started",
        "timestamp": datetime.now().isoformat(timespec="seconds"),
    })

    activity.append("Quiz ready. Select an answer for each question.")
    return memory, activity


def quiz_markdown(quiz):
    """Display questions without exposing the answer key."""
    lines = ["## Practice Quiz — 5 MCQs"]

    for i, question in enumerate(quiz, start=1):
        lines.append(f"### {i}. {question['question']}")
        for letter in ["A", "B", "C", "D"]:
            lines.append(f"- **{letter}.** {question['options'][letter]}")
        lines.append("")

    return "\n".join(lines)


def results_markdown(results):
    """Display the verified score and answer review."""
    lines = [
        "## Quiz Results",
        f"**Score:** {results['score']}/5",
        f"**Percentage:** {results['percentage']}%",
        "",
        "### Answer Review",
    ]

    for item in results["question_results"]:
        status = "Correct" if item["is_correct"] else "Incorrect"
        lines.extend([
            f"**Question {item['number']} — {status}**",
            f"- Topic: {item['topic']}",
            f"- Your answer: {item['submitted']}",
            f"- Correct answer: {item['correct_answer']}",
            f"- Explanation: {item['explanation']}",
            "",
        ])

    if results["weak_areas"]:
        lines.append("### Topics to Review")
        lines.extend(f"- {area}" for area in results["weak_areas"])
    else:
        lines.append("### All five answers are correct!")

    return "\n".join(lines)


print("Main workflow and display utilities are ready.")

Main workflow and display utilities are ready.


In [ ]:
# Cell 12: Gradio callbacks for session, evaluation and re-teaching

def ui_start(name, topic, difficulty):
    """Start a tutoring session from the Gradio interface."""
    try:
        memory, activity = start_session(name, topic, difficulty)

        return (
            memory,
            "\n".join(activity),
            f"## Learning Plan\n\n{memory['learning_plan']}",
            f"## Lesson\n\n{memory['lesson']}",
            quiz_markdown(memory["quiz"]),
            *[gr.update(choices=["A", "B", "C", "D"], value=None)
              for _ in range(5)],
            "",
            "",
            "Session ready. Answer all five questions.",
        )

    except Exception as error:
        message = f"{type(error).__name__}: {str(error)[:500]}"
        return (
            new_memory(),
            f"Session error: {message}",
            "",
            "",
            "",
            *[gr.update(choices=["A", "B", "C", "D"], value=None)
              for _ in range(5)],
            "",
            "",
            "Could not start the session. Check the error and try again.",
        )


def ui_submit(memory, a1, a2, a3, a4, a5):
    """Grade answers and request feedback from the Evaluator Agent."""
    try:
        if not memory or not memory.get("quiz"):
            raise ValueError("Start a learning session first.")

        answers = [a1, a2, a3, a4, a5]

        if any(answer not in {"A", "B", "C", "D"} for answer in answers):
            raise ValueError("Please answer all five questions.")

        results = score_quiz(memory["quiz"], answers)

        feedback = evaluate_answers(
            memory["student_name"],
            memory["topic"],
            memory["lesson"],
            results,
        )

        memory.update({
            "answers": answers,
            "score": results["score"],
            "feedback": feedback,
            "weak_areas": results["weak_areas"],
        })

        memory["history"].append({
            "event": "quiz_evaluated",
            "score": results["score"],
            "timestamp": datetime.now().isoformat(timespec="seconds"),
        })

        activity = (
            "Evaluator completed.\n"
            f"Verified score: {results['score']}/5\n"
            f"Weak areas: {results['weak_areas'] or 'None'}"
        )

        return (
            memory,
            activity,
            results_markdown(results),
            f"## Evaluator Feedback\n\n{feedback}",
            "Evaluation completed.",
        )

    except Exception as error:
        message = f"{type(error).__name__}: {str(error)[:500]}"
        return memory, f"Evaluation error: {message}", "", "", message


def ui_reteach(memory):
    """Run targeted re-teaching based on evaluated weak areas."""
    try:
        if not memory or memory.get("score") is None:
            raise ValueError("Submit your quiz before requesting re-teaching.")

        if memory["reteach_attempts"] >= MAX_RETEACH_ATTEMPTS:
            raise ValueError("Maximum re-teaching attempts reached for this session.")

        results = score_quiz(memory["quiz"], memory["answers"])

        reteaching = reteach_weak_areas(
            memory["student_name"],
            memory["topic"],
            memory["weak_areas"],
            results["question_results"],
        )

        memory["reteaching"] = reteaching
        memory["reteach_attempts"] += 1
        memory["history"].append({
            "event": "reteaching",
            "attempt": memory["reteach_attempts"],
            "timestamp": datetime.now().isoformat(timespec="seconds"),
        })

        return (
            memory,
            f"Re-teaching completed (attempt {memory['reteach_attempts']}).",
            f"## Targeted Re-teaching\n\n{reteaching}",
            "Review the targeted lesson. Start a new session to practice again.",
        )

    except Exception as error:
        message = f"{type(error).__name__}: {str(error)[:500]}"
        return memory, message, "", message


print("Gradio callbacks are ready.")

Gradio callbacks are ready.


In [ ]:
# Cell 13: Build the Leo AI Tutor user interface

with gr.Blocks(
    title=APP_NAME,
    theme=gr.themes.Soft(),
) as demo:

    gr.Markdown(
        """
        # Leo — Multi-Agent AI Tutor
        **Learn · Practice · Evaluate · Improve**

        A multi-agent learning assistant powered by CrewAI and
        Gemini 3.1 Flash-Lite.
        """
    )

    memory_state = gr.State(new_memory())

    with gr.Row():
        name_input = gr.Textbox(
            label="Student Name",
            placeholder="Enter your name",
        )
        topic_input = gr.Textbox(
            label="Learning Topic",
            placeholder="e.g. Python, RAG, Machine Learning",
        )

    difficulty_input = gr.Radio(
        ["Beginner", "Intermediate", "Advanced"],
        value="Beginner",
        label="Difficulty",
    )

    start_button = gr.Button("Start Learning Session", variant="primary")
    activity_output = gr.Textbox(
        label="Agent Activity Log",
        lines=4,
        interactive=False,
    )

    with gr.Accordion("Coordinator — Learning Plan", open=True):
        plan_output = gr.Markdown()

    with gr.Accordion("Explainer — Lesson", open=True):
        lesson_output = gr.Markdown()

    quiz_output = gr.Markdown()

    gr.Markdown("## Submit Your Answers")

    answer_inputs = [
        gr.Radio(
            choices=["A", "B", "C", "D"],
            label=f"Question {i} Answer",
        )
        for i in range(1, 6)
    ]

    submit_button = gr.Button("Submit Answers", variant="primary")

    results_output = gr.Markdown()
    feedback_output = gr.Markdown()
    status_output = gr.Textbox(label="Status", interactive=False)

    reteach_button = gr.Button("Re-teach My Weak Areas")
    reteach_output = gr.Markdown()

    start_button.click(
        fn=ui_start,
        inputs=[name_input, topic_input, difficulty_input],
        outputs=[
            memory_state,
            activity_output,
            plan_output,
            lesson_output,
            quiz_output,
            *answer_inputs,
            results_output,
            feedback_output,
            status_output,
        ],
    )

    submit_button.click(
        fn=ui_submit,
        inputs=[memory_state, *answer_inputs],
        outputs=[
            memory_state,
            activity_output,
            results_output,
            feedback_output,
            status_output,
        ],
    )

    reteach_button.click(
        fn=ui_reteach,
        inputs=[memory_state],
        outputs=[
            memory_state,
            activity_output,
            reteach_output,
            status_output,
        ],
    )

print("Leo interface created.")

/tmp/ipykernel_565/1604198193.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Leo interface created.


In [ ]:
# Cell 14: Launch the Gradio application

demo.queue(default_concurrency_limit=1)

demo.launch(
    share=True,
    debug=False,
    show_error=True,
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a7ced6e211e4e18d8c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
